### Загрузка

Импорты

In [233]:
from geopy.distance import geodesic
import osmnx as ox
import requests
import pandas as pd
import numpy as np
import gpxpy
from ipywidgets import IntProgress
from IPython.display import display

Для определения расстояния между соседними точками используется функция length, основанная на вычислении геодезического расстояния с помощью библиотеки geopy.

In [234]:
links = []
with open("Links.txt", mode="r", encoding="UTF-8") as f:
	for i in f:
		links.append(i.strip())

In [235]:
links

['https://caucasia.ru/gpx/8EWQAV62.gpx',
 'https://caucasia.ru/gpx/CPOQQN9R.gpx',
 'https://caucasia.ru/gpx/CVBOS4W2.gpx',
 'https://caucasia.ru/gpx/HTGT4UOT.gpx',
 'https://caucasia.ru/gpx/HQUWT5UB.gpx',
 'https://caucasia.ru/gpx/S87VDJ9R.gpx',
 'https://caucasia.ru/gpx/XKSRMH5L.gpx',
 'https://caucasia.ru/gpx/S87VDJ9R.gpx',
 'https://caucasia.ru/gpx/G6I1K7J9.gpx']

In [ ]:
"""
CREATW TABLE analysis_track (
    track_id VARCHAR(255) NOT NULL,         -- Номер трека
    latitude VARCHAR(255) NOT NULL,           -- Широта
    longitude VARCHAR(255),                     -- Долгота
    elevation INT,                       -- высота над уровнем моря
    time TIMESTAMP,                     -- Дата
    step_frec INT,                       -- Частота шагов
    temperature VARCHAR(500)                  -- Температура
	
);
"""

записываю треки в файлы

In [236]:
for num, url in enumerate(links):
	response = requests.get(url)
	filename = f"data/gpx/track{num}.gpx"
	
	with open(filename, "wb") as f:
		f.write(response.content)

получаю данные по трекам

In [237]:
result = []

for i in range(9):
    gpx_file = open(f'data/gpx/track{i}.gpx', 'r')
    gpx = gpxpy.parse(gpx_file)    
    
    # Process points and update bar
    for track in gpx.tracks:
        for segment in track.segments:
            for point in segment.points:
                result.append({"track_id":i,
                               "latitude":point.latitude, 
                               "longitude":point.longitude,
                               "elevation":point.elevation,
                               "time":point.time.date()})

### Парсинг данных

In [238]:
df = pd.DataFrame()

In [239]:
parsed_gpx = pd.DataFrame(result)
parsed_gpx

,track_id,latitude,longitude,elevation,time
0,0,44.589199,38.400221,187.926320,2025-11-17
1,0,44.589160,38.400190,188.159584,2025-11-17
2,0,44.589230,38.400040,188.616480,2025-11-17
3,0,44.589560,38.399690,189.506496,2025-11-17
4,0,44.590050,38.399120,188.224640,2025-11-17
...,...,...,...,...,...
32001,8,44.736280,37.772490,0.610000,2006-11-26
32002,8,44.736330,37.771890,43.920000,2006-11-26
32003,8,44.736140,37.773240,5.420000,2006-11-26
32004,8,44.736260,37.771970,5.420000,2006-11-26


In [240]:
def length(parsed_gpx):
    """
    расчёт расстояний между соседними точками используя geopy
    """
    length_3d = []
    for track in parsed_gpx.groupby('track_id'):
        track = track[1].sort_values('time')
        for idx in track.index[:-1]:
            p1 = (track['latitude'][idx], track['longitude'][idx])
            p2 = (track['latitude'][idx + 1], track['longitude'][idx + 1])
            length_2d = geodesic(p1, p2).meters
            if pd.notna(track['elevation'][idx]):
                length_track = np.sqrt((length_2d ** 2) + (track['elevation'][idx] - track['elevation'][idx + 1]) ** 2)
            else:
                length_track = length_2d
            length_3d.append(length_track)
        length_3d.append(None)
    return length_3d

In [241]:
parsed_gpx = pd.DataFrame(result)
parsed_gpx['time'] = pd.to_datetime(parsed_gpx['time'])

# df["length"] = length(parsed_gpx)


In [242]:
def mean_day_temp(lat, lon, time):
    """
    Шаблон Get запроса к open-meteo для получения температуры в указанный день
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
     "latitude": lat,
     "longitude": lon,
     "start_date": time,
     "end_date": time,
     "daily": "temperature_2m_mean",
    }
    response = requests.get(url, params=params)
    return response.json()['daily']['temperature_2m_mean']

In [243]:
def temp_per_track(parsed_gpx):
    """
    Расчёт среддней температуры для каждого трека
    """
    temp = []
    for track in parsed_gpx.groupby('track_id'):
        track = track[1]
        if pd.notna(track['time']).sum():
            id = track.index[0]
            test = mean_day_temp(track['latitude'][id], track['longitude'][id], str(track['time'][id])[:10])
            temp.append(test[0])
        else:
            temp.append(None)
    return temp

In [244]:
def parse_features(lat, lon, radius=500):
    """
    Подсчёт количества обьектов на заданном расстояние от точки с использованием osmnx
    """
    point = (lat, lon)
    tags = {
        "water": {"natural": "water"},
        "forest": {"natural": "wood"},
        "buildings": {"building": True}
    }
    
    features = {}
    for key, tag in tags.items():
        try:
            gdf = ox.features.features_from_point(point, dist=radius, tags=tag)
            features[key] = len(gdf)
        except Exception as e:
            features[key] = 0
    
    return features

In [245]:
# temp_date = {}
# object_date = {}

# for date in parsed_gpx["time"].dt.date.unique():
#     date_data = parsed_gpx[parsed_gpx["time"].dt.date == date].iloc[0]
#     lat = date_data["latitude"]
#     lon = date_data["longitude"]
    
#     temp_date[date] = mean_day_temp(lat, lon, date)[0]
#     object_date[date] = parse_features(lat, lon)
    
#     print(f"Данные по дате: {date} получены")

# parsed_gpx["temperature"] = parsed_gpx["time"].map(temp_date)
# parsed_gpx["objects"] = parsed_gpx["time"].map(object_date)


In [246]:
def get_features(parsed_gpx, n=1000):
    """
    Получение информации о количестве обьектов вокруг каждой n точки (с целью оптимизации)
    """
    bar = IntProgress(min=0, max=len(parsed_gpx.index))
    display(bar)
    water = []
    forest = []
    buildings = []
    features = parse_features(parsed_gpx['latitude'][0], parsed_gpx['longitude'][0])
    for idx in parsed_gpx.index:
        if idx % n == 0:
            features = parse_features(parsed_gpx['latitude'][idx], parsed_gpx['longitude'][idx])
        water.append(features['water'])
        forest.append(features['forest'])
        buildings.append(features['buildings'])
        bar.value += 1
    return water, forest, buildings

In [247]:
def around_type(df):
    """
    Определение типа местности на основе информации о количестве обьектов вокруг точки
    """
    if pd.notna(df[['water', 'forest', 'buildings']]).sum() and df[['water', 'forest', 'buildings' ]].sum():
        if df['water'] + df['forest'] > 0:
            around = {'water': df['water'], 'forest': df['forest']}
            return max(around, key=around.get)
        if df['buildings'] > 0:
            return 'city'
    return None

In [248]:
# 1. Расчёт расстояний между точками
print("Расчёт расстояний между точками...")
length_3d = length(parsed_gpx)
parsed_gpx['distance_3d'] = length_3d  # Убираем последний None

# 2. Расчёт средней температуры для каждого трека
print("Расчёт средней температуры для каждого трека...")
temperatures = temp_per_track(parsed_gpx)

# Создаем серию температур с правильным соответствием индексам
temp_series = pd.Series(temperatures, index=parsed_gpx.groupby('track_id').groups.keys())
parsed_gpx['mean_temperature'] = parsed_gpx['track_id'].map(temp_series)

# 3. Получение информации об объектах вокруг точек
print("Получение информации об объектах вокруг точек...")
print("Этот процесс может занять некоторое время...")
water, forest, buildings = get_features(parsed_gpx)

parsed_gpx['water_count'] = water
parsed_gpx['forest_count'] = forest
parsed_gpx['buildings_count'] = buildings

# 4. Определение типа местности
print("Определение типа местности...")
around_type_results = []
for idx, row in parsed_gpx.iterrows():
    # Создаем временный датафрейм с нужными колонками
    temp_df = pd.DataFrame({
        'water': [row['water_count']],
        'forest': [row['forest_count']],
        'buildings': [row['buildings_count']]
    }).iloc[0]
    
    result = around_type(temp_df)
    around_type_results.append(result)

parsed_gpx['area_type'] = around_type_results


# Выводим информацию о результате
print("\n" + "="*50)
print("ОБНОВЛЕНИЕ ЗАВЕРШЕНО!")
print("="*50)
print(f"Добавленные колонки:")
print(f"- distance_3d: расстояние до следующей точки (3D)")
print(f"- mean_temperature: средняя температура для трека")
print(f"- water_count: количество водных объектов в радиусе 500м")
print(f"- forest_count: количество лесных объектов в радиусе 500м")
print(f"- buildings_count: количество зданий в радиусе 500м")
print(f"- area_type: тип местности (water/forest/city/None)")
print(f"\nРазмер датафрейма: {parsed_gpx.shape}")
print(f"Количество уникальных треков: {parsed_gpx['track_id'].nunique()}")

# Сохраняем обновленный датафрейм (опционально)
# parsed_gpx.to_csv('parsed_gpx_updated.csv', index=False)
# print("Датафрейм сохранен в файл 'parsed_gpx_updated.csv'")

# Показываем первые строки обновленного датафрейма
print("\nПервые 5 строк обновленного датафрейма:")
print(parsed_gpx.head())

Расчёт расстояний между точками...
Расчёт средней температуры для каждого трека...
Получение информации об объектах вокруг точек...
Этот процесс может занять некоторое время...


IntProgress(value=0, max=32006)

KeyboardInterrupt: 